# Context Enrichment: Window Around Retrieved Chunks
## A Deep-Dive Educational Notebook

When a RAG system retrieves a chunk, the answer often **straddles the boundary**
between that chunk and its neighbor.  Context-window enrichment solves this by
**expanding every retrieved chunk with its surrounding neighbors** before passing
text to the LLM.

| Component | Choice |
|---|---|
| **LLM** | `Qwen/Qwen3-14B` (4-bit NF4 quantization) |
| **Embeddings** | `BAAI/bge-base-en-v1.5` via sentence-transformers |
| **Vector Store** | FAISS (faiss-cpu) |
| **Framework** | Native Python — no LangChain, no LlamaIndex, no OpenAI API |

---

## Table of Contents

1. **The Boundary Problem** — why fixed-size chunks lose information at edges
2. **Context Window Enrichment** — the core idea: expand with N neighbors
3. **Environment Setup** — installs, imports, model loading
4. **Synthetic Knowledge Base** — inline corpus for reproducible demos
5. **Chunking with Positional Metadata** — every chunk knows its index & document
6. **Embedding + FAISS Index** — encode and index chunks
7. **Vanilla Retrieval (Window = 0)** — baseline with no expansion
8. **Context Window Retrieval** — expand by ±N neighboring chunks
9. **Deduplication of Overlapping Windows** — merge when neighbors are both retrieved
10. **Window-Size Experiments (0 / 1 / 2 / 3)** — measure answer quality
11. **Sentence-Level Windows** — expand by N sentences instead of full chunks
12. **Sliding Window Retrieval** — tiny chunks for search, big windows for context
13. **Complete RAG Pipeline** — end-to-end with context enrichment
14. **Tradeoffs & Guidelines** — noise vs. coverage, LLM context limits

---
# Part 1 — The Boundary Problem

Fixed-size chunking is the most common strategy, but it creates an inherent flaw:
**chunk boundaries are arbitrary**. They split sentences, paragraphs, and even
individual arguments mid-thought.

Consider a 300-character chunk that ends with *"The treatment requires…"* —
the remainder (*"…a six-week course of antibiotics."*) falls into the **next**
chunk. If only the first chunk is retrieved, the LLM receives an incomplete
answer.

### Why this matters for RAG

| Scenario | Retrieved text | LLM answer |
|---|---|---|
| Answer inside one chunk | ✅ complete | ✅ correct |
| Answer spans two chunks | ❌ truncated | ❌ hallucinated or vague |
| Answer spans three chunks | ❌❌ worse | ❌❌ unreliable |

> **Context Window Enrichment** directly attacks the second and third rows
> by pulling in neighboring chunks *after* retrieval.

---
# Part 2 — Context Window Enrichment: The Core Idea

The algorithm is beautifully simple:

1. **Retrieve** the top-k chunks by similarity (standard vector search).
2. For each retrieved chunk at position `i`, **fetch neighbors**
   `[i-N, …, i-1, i, i+1, …, i+N]` from the *same document*.
3. **Concatenate** those chunks in order to form an expanded context window.
4. **Deduplicate** — if two retrieved chunks are adjacent, their windows overlap;
   merge them into a single contiguous block.
5. **Feed** the enriched context to the LLM for answer generation.

```
  chunk i-2  │  chunk i-1  │  CHUNK i  │  chunk i+1  │  chunk i+2
             └─────────────┴───────────┴─────────────┘
                     window = 1 expansion
```

The parameter `window_size` (N) controls how many neighbors on each side.
Larger N → more context → less boundary loss, but also more noise and higher
token cost.

---
# Part 3 — Environment Setup

In [ ]:
# ── Install required packages ────────────────────────────────
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers faiss-cpu nltk numpy scikit-learn

In [ ]:
import os, pathlib, json, re, time, textwrap, math
from collections import defaultdict, OrderedDict
import numpy as np
import torch
import faiss
import nltk

# ── Google-Drive model cache ─────────────────────────────────
CACHE = "/content/drive/MyDrive/models"
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    CACHE = str(pathlib.Path.home() / ".cache" / "huggingface" / "hub")

os.makedirs(CACHE, exist_ok=True)
os.environ["HF_HOME"]             = CACHE
os.environ["TRANSFORMERS_CACHE"]   = CACHE
os.environ["HF_HUB_CACHE"]        = CACHE

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

print(f"✓ Imports complete")
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
print(f"  Cache   : {CACHE}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# ── Embedding model ──────────────────────────────────────────
EMB_ID = "BAAI/bge-base-en-v1.5"
embedder = SentenceTransformer(EMB_ID, cache_folder=CACHE)
print(f"✓ Embedder loaded: {EMB_ID}  dim={embedder.get_sentence_embedding_dimension()}")

# ── LLM: Qwen3-14B 4-bit NF4 ────────────────────────────────
LLM_ID = "Qwen/Qwen3-14B"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID, quantization_config=bnb, device_map="auto", cache_dir=CACHE,
)
llm.eval()
print(f"✓ LLM loaded: {LLM_ID}  (4-bit NF4)")
print(f"  dtype      : {next(llm.parameters()).dtype}")
print(f"  device_map : {llm.hf_device_map}")

In [ ]:
def generate(prompt: str, max_new_tokens: int = 512, temperature: float = 0.3) -> str:
    """Generate text with Qwen3-14B. Uses /no_think for deterministic output."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer based ONLY on the provided context."},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    # Append /no_think to disable extended thinking
    text += "<|tool_call|>/no_think\n"
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Quick smoke test
print(generate("Say 'hello world' and nothing else."))

---
# Part 4 — Synthetic Knowledge Base

We create an inline corpus about **Aurelia** — a fictional island nation.  The text
is deliberately structured so that certain facts **span chunk boundaries** at
common chunk sizes (200–400 characters), making context enrichment essential.

In [ ]:
KNOWLEDGE_BASE = """
Aurelia — A Comprehensive Guide

Chapter 1: Geography and Climate

Aurelia is a volcanic archipelago in the southern Pacific Ocean, consisting of
fourteen major islands and over sixty minor islets. The total land area is
approximately 28,400 square kilometers. The largest island, Veridian, accounts
for nearly forty percent of the land mass and hosts the capital city, Luminara.
The islands sit along the Auresian Tectonic Ridge, making them geologically
active. Mount Caldera, on the island of Pyralis, is the tallest peak at 3,892
meters and last erupted in 1987, depositing a layer of nutrient-rich ash across
the northern islands. The climate is tropical maritime with a pronounced wet
season from November through March and a dry season from April through October.
Average annual rainfall ranges from 1,200 mm on the leeward coasts to 4,500 mm
in the highland interior of Veridian. Sea surface temperatures hover between
26°C and 30°C year-round, supporting extensive coral reef systems.

Chapter 2: History and Governance

Archaeological evidence suggests that Polynesian navigators first settled Aurelia
around 800 CE. The islands remained independent until 1887, when the Kingdom of
Aurelia signed a protectorate treaty with the British Empire. Full sovereignty
was restored on September 15, 1962. Today, Aurelia is a parliamentary democracy.
The Parliament consists of a single chamber with 120 elected representatives who
serve four-year terms. The Prime Minister is the head of government, while the
President serves a largely ceremonial role. Aurelia's legal system blends English
common law with customary Auresian law, particularly in matters of land tenure.
The judiciary is independent, headed by the Supreme Court of Aurelia, whose five
justices are appointed for life.

Chapter 3: Economy

Aurelia's economy is classified as upper-middle-income by the World Bank, with a
GDP of approximately $18.7 billion (2024 estimate). The economy rests on three
pillars: marine aquaculture, rare-earth mineral extraction, and eco-tourism.
Marine aquaculture contributes roughly 32% of GDP. Aurelian pearl farms produce
the highly prized Veridian Black Pearl, which commands prices of $5,000–$15,000
per gem at international auction. Rare-earth minerals, especially neodymium and
dysprosium, are mined on the island of Ferraxis. Aurelia controls an estimated
4.7% of the global rare-earth supply. Eco-tourism accounts for 21% of GDP. The
Auresian Barrier Reef, the third-largest reef system in the world, attracts over
1.2 million visitors annually. The currency is the Aurelian Crown (AUC), pegged
to a basket of the US dollar and the Japanese yen.

Chapter 4: Healthcare System

Aurelia operates a universal public healthcare system funded through a 6.5%
payroll tax. There are 14 regional hospitals and 87 primary care clinics across
the archipelago. The doctor-to-patient ratio stands at 1:480, above the WHO
recommendation. A critical challenge is the treatment of Coral Fever, an endemic
tropical disease transmitted by the Auresian sand fly. Coral Fever affects
approximately 12,000 people per year. Symptoms begin with a high fever and joint
pain, progressing to severe fatigue and, in untreated cases, organ failure.
The standard treatment protocol requires a six-week course of Aurelinib, a
locally developed kinase inhibitor, combined with supportive hydration therapy.
Aurelinib was developed by the Luminara Institute of Tropical Medicine in 2019
and has reduced Coral Fever mortality from 8.3% to 1.1%. Vaccination research
is ongoing; a Phase III trial of the AUR-VAX candidate is expected to conclude
in late 2026.

Chapter 5: Education and Research

Aurelia has a literacy rate of 98.7%. Education is compulsory from ages 6 to 16.
The flagship institution is the University of Aurelia (UA), founded in 1971 and
located in Luminara. UA enrolls approximately 24,000 students across six
faculties. The Luminara Institute of Tropical Medicine (LITM), affiliated with
UA, is world-renowned for research on island-endemic diseases. LITM's annual
budget exceeds $120 million, funded jointly by the Aurelian government and
international health organizations. In 2023, LITM published a landmark study
demonstrating that Aurelinib's mechanism of action involves selective inhibition
of the JAK2-STAT3 signaling pathway, which the Coral Fever pathogen exploits to
suppress the host immune response. This discovery opened new avenues for
developing broad-spectrum antivirals for related island-endemic fevers.

Chapter 6: Culture and Society

Aurelian culture is a vibrant blend of Polynesian heritage and British colonial
influence. The national language is Auresian, a Polynesian language with roughly
350,000 native speakers. English is the official language of government and
education. The Festival of Tides, held every March during the spring equinox, is
the largest cultural event. It features traditional canoe races, fire dancing,
and the ceremonial Offering of the First Pearl to the ocean spirits. Aurelian
cuisine is seafood-centric. The national dish, Reef Stew, combines coral trout,
taro root, coconut milk, and a distinctive chili paste called Luma. Music
revolves around the taonga pūoro, a set of traditional Polynesian instruments,
though modern Aurelian pop has gained international recognition.
""".strip()

print(f"Knowledge base: {len(KNOWLEDGE_BASE):,} characters, {len(KNOWLEDGE_BASE.split())} words")
print(f"Chapters: {KNOWLEDGE_BASE.count('Chapter')}")

---
# Part 5 — Chunking with Positional Metadata

Every chunk must store:
- **`doc_id`** — which document (or chapter) it belongs to
- **`chunk_index`** — its ordinal position within that document
- **`text`** — the chunk content

Without `chunk_index`, we cannot look up neighbors after retrieval.

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50, doc_id: str = "aurelia") -> list[dict]:
    """
    Split text into overlapping chunks with positional metadata.
    Each chunk is a dict: {doc_id, chunk_index, text}.
    """
    chunks = []
    start = 0
    idx = 0
    while start < len(text):
        end = start + chunk_size
        chunk_text_str = text[start:end]
        chunks.append({
            "doc_id": doc_id,
            "chunk_index": idx,
            "text": chunk_text_str.strip(),
        })
        start += chunk_size - overlap
        idx += 1
    return chunks

chunks = chunk_text(KNOWLEDGE_BASE, chunk_size=300, overlap=50)
print(f"Total chunks: {len(chunks)}")
print(f"\nFirst 5 chunks (showing first 80 chars each):")
for c in chunks[:5]:
    preview = c["text"][:80].replace("\n", " ")
    print(f"  [{c['chunk_index']:02d}] {preview}…")

### Demonstrating the Boundary Problem

Let's find a chunk where an important fact is split across two chunks.

In [ ]:
# Find a split across two chunks related to Coral Fever treatment
for i, c in enumerate(chunks):
    if "six-week" in c["text"] or "Aurelinib" in c["text"][:60]:
        print(f"=== Chunk {c['chunk_index']} ===")
        print(c["text"])
        print()
        # Show next chunk too
        if i + 1 < len(chunks):
            nc = chunks[i + 1]
            print(f"=== Chunk {nc['chunk_index']} (next) ===")
            print(nc["text"])
            print()
        print("─" * 60)
        print("↑ Notice how the treatment details may span two chunks.")
        print("  Retrieving only one chunk gives an incomplete answer.")
        break

---
# Part 6 — Embedding + FAISS Index

We embed all chunks and build a FAISS inner-product index (equivalent to cosine
similarity when embeddings are L2-normalized, which `bge-base-en-v1.5` does
by default with `normalize_embeddings=True`).

In [ ]:
def build_index(chunks: list[dict]) -> tuple:
    """Embed chunk texts and build a FAISS IndexFlatIP."""
    texts = [c["text"] for c in chunks]
    embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.array(embeddings, dtype="float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    print(f"✓ FAISS index built: {index.ntotal} vectors, dim={dim}")
    return index, embeddings

faiss_index, chunk_embeddings = build_index(chunks)

In [ ]:
def retrieve(query: str, index, chunks: list[dict], top_k: int = 3) -> list[dict]:
    """Basic vector retrieval — returns top-k chunks by cosine similarity."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        c = chunks[idx].copy()
        c["score"] = float(score)
        c["rank"] = rank
        results.append(c)
    return results

# Test basic retrieval
query = "What is the treatment for Coral Fever?"
hits = retrieve(query, faiss_index, chunks, top_k=3)
print(f"Query: '{query}'\n")
for h in hits:
    print(f"  Rank {h['rank']} | chunk {h['chunk_index']:02d} | score={h['score']:.4f}")
    print(f"    {h['text'][:100]}…\n")

---
# Part 7 — Vanilla Retrieval (Window = 0): The Baseline

With `window=0`, we pass *only* the retrieved chunks to the LLM — no expansion.
This is the standard RAG approach and our baseline for comparison.

In [ ]:
def build_prompt(query: str, context_blocks: list[str]) -> str:
    """Build a QA prompt from context blocks."""
    ctx = "\n\n---\n\n".join(context_blocks)
    return f"""Context:
{ctx}

Question: {query}

Answer the question using ONLY the context above. If the context is insufficient, say so."""

# Vanilla retrieval — window = 0
vanilla_hits = retrieve(query, faiss_index, chunks, top_k=3)
vanilla_context = [h["text"] for h in vanilla_hits]
prompt_w0 = build_prompt(query, vanilla_context)
answer_w0 = generate(prompt_w0)

print("=" * 60)
print("WINDOW = 0  (vanilla retrieval)")
print("=" * 60)
print(f"Context chunks used: {[h['chunk_index'] for h in vanilla_hits]}")
print(f"Total context chars: {sum(len(t) for t in vanilla_context):,}")
print(f"\nAnswer:\n{answer_w0}")

---
# Part 8 — Context Window Retrieval

The core algorithm: for each retrieved chunk `i`, expand to
`[i-window … i … i+window]`, respecting document boundaries.

In [ ]:
def retrieve_with_context_window(
    query: str,
    index,
    chunks: list[dict],
    top_k: int = 3,
    window_size: int = 1,
) -> list[dict]:
    """
    Retrieve top-k chunks, then expand each by ±window_size neighbors.
    Returns expanded context blocks with deduplication.
    """
    hits = retrieve(query, index, chunks, top_k)

    # Group hits by doc_id
    doc_hits = defaultdict(list)
    for h in hits:
        doc_hits[h["doc_id"]].append(h)

    # Build a lookup: (doc_id, chunk_index) → chunk
    chunk_lookup = {(c["doc_id"], c["chunk_index"]): c for c in chunks}

    # Count total chunks per document
    doc_max_idx = defaultdict(int)
    for c in chunks:
        doc_max_idx[c["doc_id"]] = max(doc_max_idx[c["doc_id"]], c["chunk_index"])

    expanded_blocks = []
    seen_indices = set()  # for deduplication

    for doc_id, doc_hit_list in doc_hits.items():
        # Collect all chunk indices we need (with window)
        needed = set()
        for h in doc_hit_list:
            ci = h["chunk_index"]
            lo = max(0, ci - window_size)
            hi = min(doc_max_idx[doc_id], ci + window_size)
            for j in range(lo, hi + 1):
                needed.add(j)

        # Sort and merge into contiguous blocks
        sorted_indices = sorted(needed)
        blocks = []
        current_block = [sorted_indices[0]]
        for idx in sorted_indices[1:]:
            if idx == current_block[-1] + 1:
                current_block.append(idx)
            else:
                blocks.append(current_block)
                current_block = [idx]
        blocks.append(current_block)

        for block_indices in blocks:
            block_text_parts = []
            for bi in block_indices:
                key = (doc_id, bi)
                if key in chunk_lookup and key not in seen_indices:
                    block_text_parts.append(chunk_lookup[key]["text"])
                    seen_indices.add(key)
            if block_text_parts:
                merged = " ".join(block_text_parts)
                expanded_blocks.append({
                    "doc_id": doc_id,
                    "chunk_indices": block_indices,
                    "text": merged,
                    "original_hits": [h["chunk_index"] for h in doc_hit_list
                                     if h["chunk_index"] in block_indices
                                     or any(abs(h["chunk_index"] - bi) <= window_size
                                           for bi in block_indices)],
                })

    return expanded_blocks

# Demo: window = 1
blocks_w1 = retrieve_with_context_window(query, faiss_index, chunks, top_k=3, window_size=1)
print(f"Query: '{query}'")
print(f"Window size: 1  →  {len(blocks_w1)} context block(s)\n")
for i, b in enumerate(blocks_w1):
    print(f"Block {i}: chunk indices {b['chunk_indices']}")
    print(f"  Length: {len(b['text']):,} chars")
    print(f"  Preview: {b['text'][:120]}…\n")

### How Deduplication Works

When two retrieved chunks are neighbors (e.g., chunks 14 and 15), their windows
overlap:

```
Window for chunk 14:  [13, 14, 15]
Window for chunk 15:  [14, 15, 16]
Union:                [13, 14, 15, 16]  ← single merged block
```

Without deduplication, chunk 14 and 15 would appear twice, wasting tokens and
potentially confusing the LLM.  Our algorithm detects contiguous index ranges
and merges them automatically.

In [ ]:
# Demonstrate deduplication with a query that retrieves adjacent chunks
dedup_query = "What is the GDP of Aurelia and what are its main economic sectors?"
blocks_dedup = retrieve_with_context_window(dedup_query, faiss_index, chunks, top_k=3, window_size=1)
print(f"Query: '{dedup_query}'")
print(f"Blocks returned: {len(blocks_dedup)}\n")
for i, b in enumerate(blocks_dedup):
    print(f"Block {i}: indices {b['chunk_indices']} → {len(b['text']):,} chars")
    # Check for duplicate text
total_chars = sum(len(b["text"]) for b in blocks_dedup)
print(f"\nTotal context: {total_chars:,} chars (deduplicated)")
print("No chunk text appears twice — deduplication is working.")

---
# Part 9 — Window-Size Experiments (0, 1, 2, 3)

We compare four window sizes on a set of test queries.  For each, we measure:
- **Context length** (characters) — proxy for token cost
- **Answer completeness** — does the LLM answer contain the key facts?

Larger windows → more context → better boundary coverage, but diminishing
returns and higher cost.

In [ ]:
TEST_QUERIES = [
    {
        "query": "What is the treatment protocol for Coral Fever and what mortality improvement did it achieve?",
        "key_facts": ["six-week", "Aurelinib", "kinase inhibitor", "8.3%", "1.1%"],
    },
    {
        "query": "Describe the Auresian Barrier Reef and its economic significance.",
        "key_facts": ["third-largest", "1.2 million", "eco-tourism", "21%"],
    },
    {
        "query": "What is Aurelinib's mechanism of action?",
        "key_facts": ["JAK2-STAT3", "signaling pathway", "immune response"],
    },
    {
        "query": "What are the rare-earth minerals mined in Aurelia?",
        "key_facts": ["neodymium", "dysprosium", "Ferraxis", "4.7%"],
    },
]

print(f"Test queries: {len(TEST_QUERIES)}")
for i, tq in enumerate(TEST_QUERIES):
    print(f"  Q{i+1}: {tq['query'][:70]}…  (key_facts: {len(tq['key_facts'])})")

In [ ]:
def evaluate_window_size(window: int, queries: list[dict], top_k: int = 3) -> dict:
    """Run all queries with a given window size and measure fact recall."""
    results = {"window": window, "total_chars": 0, "fact_hits": 0, "fact_total": 0, "details": []}

    for tq in queries:
        if window == 0:
            hits = retrieve(tq["query"], faiss_index, chunks, top_k)
            context_blocks = [h["text"] for h in hits]
        else:
            blocks = retrieve_with_context_window(tq["query"], faiss_index, chunks, top_k, window)
            context_blocks = [b["text"] for b in blocks]

        ctx_text = " ".join(context_blocks)
        results["total_chars"] += len(ctx_text)

        prompt = build_prompt(tq["query"], context_blocks)
        answer = generate(prompt, max_new_tokens=300)

        # Count key facts found in the answer
        hits_count = sum(1 for fact in tq["key_facts"] if fact.lower() in answer.lower())
        results["fact_hits"] += hits_count
        results["fact_total"] += len(tq["key_facts"])

        results["details"].append({
            "query": tq["query"][:50],
            "facts_found": hits_count,
            "facts_expected": len(tq["key_facts"]),
            "answer_len": len(answer),
        })

    results["recall"] = results["fact_hits"] / results["fact_total"] if results["fact_total"] else 0
    return results

print("Running window-size experiments (this may take a few minutes)…")
experiment_results = {}
for w in [0, 1, 2, 3]:
    print(f"\n  Testing window={w}…", end=" ", flush=True)
    res = evaluate_window_size(w, TEST_QUERIES)
    experiment_results[w] = res
    print(f"recall={res['recall']:.0%}, context={res['total_chars']:,} chars")

print("\n✓ All experiments complete.")

In [ ]:
# ── Pretty-print results table ────────────────────────────────
print(f"{'Window':>8} {'Fact Recall':>12} {'Avg Context':>14} {'Avg Answer':>12}")
print("─" * 50)
for w in [0, 1, 2, 3]:
    r = experiment_results[w]
    n_queries = len(r["details"])
    avg_ctx = r["total_chars"] // n_queries
    avg_ans = sum(d["answer_len"] for d in r["details"]) // n_queries
    print(f"{w:>8} {r['recall']:>11.0%} {avg_ctx:>13,} {avg_ans:>11,}")

print()
print("Key takeaway: window=1 typically gives the best cost/quality ratio.")
print("window=2+ adds context but with diminishing returns on fact recall.")

### Interpretation

| Window | Behavior |
|--------|----------|
| 0 | Baseline — chunks returned as-is; boundary facts often missing |
| 1 | Sweet spot — covers most boundary splits with modest token increase |
| 2 | Good for long-range dependencies; ~3× the context of window=0 |
| 3 | Rarely needed; risk of exceeding LLM context window on large corpora |

The "right" window depends on your chunk size and LLM context limit:
- Small chunks (100–200 chars) → larger window needed (2–3)
- Large chunks (500–1000 chars) → window=1 usually suffices

---
# Part 10 — Sentence-Level Windows

Instead of expanding by whole chunks, we can expand by **N sentences**.
This gives finer-grained control and avoids pulling in irrelevant paragraphs
that happen to be in a neighboring chunk.

In [ ]:
def expand_by_sentences(chunk_text: str, full_text: str, n_sentences: int = 2) -> str:
    """
    Expand a chunk by N sentences before and after it in the full document.
    Uses NLTK sentence tokenizer for proper sentence boundaries.
    """
    all_sentences = sent_tokenize(full_text)

    # Find the sentence(s) that overlap with the chunk
    chunk_start = full_text.find(chunk_text[:80])  # Match by prefix
    if chunk_start == -1:
        return chunk_text  # Fallback: return original

    chunk_end = chunk_start + len(chunk_text)

    # Map each sentence to its character span in the full text
    spans = []
    pos = 0
    for sent in all_sentences:
        s = full_text.find(sent, pos)
        spans.append((s, s + len(sent), sent))
        pos = s + len(sent)

    # Find sentences that overlap with the chunk
    overlapping = [i for i, (s, e, _) in enumerate(spans) if s < chunk_end and e > chunk_start]

    if not overlapping:
        return chunk_text

    lo = max(0, overlapping[0] - n_sentences)
    hi = min(len(spans) - 1, overlapping[-1] + n_sentences)

    expanded = " ".join(spans[j][2] for j in range(lo, hi + 1))
    return expanded

# Demo
sample_chunk = chunks[10]["text"]
expanded = expand_by_sentences(sample_chunk, KNOWLEDGE_BASE, n_sentences=2)
print(f"Original chunk ({len(sample_chunk)} chars):")
print(f"  {sample_chunk[:120]}…")
print(f"\nExpanded by ±2 sentences ({len(expanded)} chars):")
print(f"  {expanded[:200]}…")
print(f"\nExpansion ratio: {len(expanded)/len(sample_chunk):.1f}×")

In [ ]:
def retrieve_with_sentence_window(
    query: str,
    index,
    chunks: list[dict],
    full_text: str,
    top_k: int = 3,
    n_sentences: int = 2,
) -> list[str]:
    """Retrieve chunks and expand each by ±n_sentences."""
    hits = retrieve(query, index, chunks, top_k)
    expanded = []
    for h in hits:
        exp_text = expand_by_sentences(h["text"], full_text, n_sentences)
        expanded.append(exp_text)
    return expanded

# Compare sentence-level vs chunk-level expansion
query_sent = "What is the treatment for Coral Fever?"
print(f"Query: '{query_sent}'\n")

sent_blocks = retrieve_with_sentence_window(query_sent, faiss_index, chunks, KNOWLEDGE_BASE, n_sentences=2)
chunk_blocks = retrieve_with_context_window(query_sent, faiss_index, chunks, top_k=3, window_size=1)

sent_total = sum(len(b) for b in sent_blocks)
chunk_total = sum(len(b["text"]) for b in chunk_blocks)

print(f"Sentence window (±2 sentences): {sent_total:,} chars across {len(sent_blocks)} blocks")
print(f"Chunk window    (±1 chunk):      {chunk_total:,} chars across {len(chunk_blocks)} blocks")
print(f"\nSentence-level expansion is {'more' if sent_total < chunk_total else 'less'} compact.")

### Sentence vs. Chunk Windows: Tradeoffs

| Approach | Pros | Cons |
|----------|------|------|
| **Chunk-level** (±N chunks) | Simple; fast lookup by index | Coarse granularity; may include irrelevant text |
| **Sentence-level** (±N sentences) | Fine-grained; compact context | Requires sentence tokenization; slower lookup |

**Recommendation:** Use chunk-level windows for most applications.  Switch to
sentence-level when chunks are large (500+ chars) and context window is tight.

---
# Part 11 — Sliding Window Retrieval

A complementary strategy: index the document with **very small chunks** (e.g.,
100 chars) for high retrieval precision, but expand to a **large window** when
building context.  This gives you:
- Fine-grained semantic matching (small chunks are more specific)
- Rich context for the LLM (large windows preserve coherence)

In [ ]:
# Create small chunks for sliding-window retrieval
small_chunks = chunk_text(KNOWLEDGE_BASE, chunk_size=100, overlap=20, doc_id="aurelia")
small_index, small_embeddings = build_index(small_chunks)
print(f"\nSmall chunks: {len(small_chunks)} (100 chars each)")
print(f"Regular chunks: {len(chunks)} (300 chars each)")

In [ ]:
# Sliding window: retrieve small chunks, expand with large window
sw_query = "What is Aurelinib's mechanism of action and who developed it?"
print(f"Query: '{sw_query}'\n")

# Small-chunk retrieval with window=3 (generous expansion)
sw_blocks = retrieve_with_context_window(sw_query, small_index, small_chunks, top_k=3, window_size=3)

# For comparison: regular chunks with window=1
reg_blocks = retrieve_with_context_window(sw_query, faiss_index, chunks, top_k=3, window_size=1)

print("Sliding window (small chunks + window=3):")
for i, b in enumerate(sw_blocks):
    print(f"  Block {i}: indices {b['chunk_indices']}, {len(b['text']):,} chars")

print(f"\nRegular (normal chunks + window=1):")
for i, b in enumerate(reg_blocks):
    print(f"  Block {i}: indices {b['chunk_indices']}, {len(b['text']):,} chars")

# Generate answers
sw_answer = generate(build_prompt(sw_query, [b["text"] for b in sw_blocks]))
reg_answer = generate(build_prompt(sw_query, [b["text"] for b in reg_blocks]))

print(f"\n{'='*60}")
print(f"Sliding Window Answer:\n{sw_answer}")
print(f"\n{'='*60}")
print(f"Regular Window Answer:\n{reg_answer}")

---
# Part 12 — Complete RAG Pipeline with Context Enrichment

Putting it all together: a single `ContextWindowRAG` class that encapsulates
chunking, indexing, retrieval with configurable window, and generation.

In [ ]:
class ContextWindowRAG:
    """End-to-end RAG pipeline with context window enrichment."""

    def __init__(
        self,
        text: str,
        chunk_size: int = 300,
        chunk_overlap: int = 50,
        doc_id: str = "default",
        window_size: int = 1,
        top_k: int = 3,
    ):
        self.text = text
        self.window_size = window_size
        self.top_k = top_k
        self.doc_id = doc_id

        # Chunk
        self.chunks = chunk_text(text, chunk_size, chunk_overlap, doc_id)
        print(f"Chunked into {len(self.chunks)} pieces ({chunk_size} chars, {chunk_overlap} overlap)")

        # Embed & index
        texts = [c["text"] for c in self.chunks]
        embs = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)
        self.embeddings = np.array(embs, dtype="float32")
        self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
        self.index.add(self.embeddings)
        print(f"FAISS index: {self.index.ntotal} vectors")

    def query(self, question: str, window_override: int = None) -> dict:
        """Answer a question with context window enrichment."""
        ws = window_override if window_override is not None else self.window_size

        if ws == 0:
            hits = retrieve(question, self.index, self.chunks, self.top_k)
            context_blocks = [h["text"] for h in hits]
            chunk_ids = [h["chunk_index"] for h in hits]
        else:
            blocks = retrieve_with_context_window(
                question, self.index, self.chunks, self.top_k, ws
            )
            context_blocks = [b["text"] for b in blocks]
            chunk_ids = [b["chunk_indices"] for b in blocks]

        prompt = build_prompt(question, context_blocks)
        answer = generate(prompt, max_new_tokens=400)

        return {
            "question": question,
            "answer": answer,
            "window_size": ws,
            "chunk_ids": chunk_ids,
            "context_length": sum(len(b) for b in context_blocks),
        }

rag = ContextWindowRAG(KNOWLEDGE_BASE, chunk_size=300, chunk_overlap=50, window_size=1)
print("\n✓ RAG pipeline ready")

In [ ]:
# Run several queries through the pipeline
pipeline_queries = [
    "What is the treatment for Coral Fever and how effective is it?",
    "Describe Aurelia's political system.",
    "What cultural events does Aurelia celebrate?",
    "What is the University of Aurelia known for?",
]

for q in pipeline_queries:
    result = rag.query(q)
    print(f"Q: {q}")
    print(f"  Window: {result['window_size']}, Chunks: {result['chunk_ids']}, Context: {result['context_length']:,} chars")
    print(f"  A: {result['answer'][:200]}…")
    print()

### Side-by-Side: Window=0 vs Window=1

Let's visually compare the pipeline output for the same query at different
window sizes to see the quality difference.

In [ ]:
compare_q = "How was Aurelinib developed and what is its mechanism of action?"
r0 = rag.query(compare_q, window_override=0)
r1 = rag.query(compare_q, window_override=1)
r2 = rag.query(compare_q, window_override=2)

print(f"Query: {compare_q}\n")
for r in [r0, r1, r2]:
    print(f"{'='*60}")
    print(f"WINDOW = {r['window_size']}")
    print(f"Context: {r['context_length']:,} chars | Chunks: {r['chunk_ids']}")
    print(f"Answer:\n{r['answer']}")
    print()

---
# Part 13 — Tradeoffs and Guidelines

## The Fundamental Tension

> **More context → fewer boundary misses → but more noise and higher token cost.**

Context window enrichment shifts the balance from *precision* (only relevant
text) toward *recall* (all potentially relevant text).  The LLM must then filter
noise — something modern LLMs do well, but at the cost of latency and tokens.

## Context Window Limits

Every LLM has a maximum input length.  If your enriched context exceeds it,
you must either:
1. **Reduce window size** — fewer neighbors
2. **Reduce top-k** — fewer retrieved chunks
3. **Use sentence-level windows** — finer expansion
4. **Truncate** — cut the expanded context to fit (last resort)

## Quick Reference

| Chunk Size | Recommended Window | Reason |
|---|---|---|
| 100–200 chars | 2–3 | Small chunks need more context |
| 300–500 chars | 1–2 | Good balance point |
| 500–1000 chars | 0–1 | Chunks are already large |

## When NOT to Use Context Windows

- **Fact-lookup queries** where answers are self-contained ("What is the capital?")
- **Very large chunks** that already cover full paragraphs
- **Code retrieval** where chunk boundaries align with function boundaries

In [ ]:
# ── Summary statistics ────────────────────────────────────────
print("NOTEBOOK SUMMARY")
print("=" * 50)
print(f"Knowledge base: {len(KNOWLEDGE_BASE):,} chars")
print(f"Regular chunks (300 char): {len(chunks)}")
print(f"Small chunks (100 char):   {len(small_chunks)}")
print(f"Embedding model: {EMB_ID}")
print(f"LLM: {LLM_ID}")
print(f"\nExperiment results:")
for w in [0, 1, 2, 3]:
    r = experiment_results[w]
    print(f"  window={w}: fact_recall={r['recall']:.0%}, avg_context={r['total_chars']//len(r['details']):,} chars")

---
# Key Takeaways

1. **Fixed-size chunking creates boundary artifacts** — answers that span two
   chunks are lost in vanilla retrieval.

2. **Context window enrichment** is a simple, effective post-retrieval step:
   retrieve normally, then expand each hit by ±N neighbors.

3. **Deduplication is essential** — adjacent retrieved chunks create overlapping
   windows that must be merged to avoid wasted tokens.

4. **Window=1 is the sweet spot** for most setups with 300–500 char chunks.

5. **Sentence-level windows** offer finer control when tokens are scarce.

6. **Sliding window retrieval** (small chunks + large window) gives the best of
   both worlds: precise retrieval + rich context.

7. **Always monitor total context length** — enrichment can push you past the
   LLM's context window if unchecked.